# `build_design.py` geometry visualization
Same design as `build_design.py`, split into cells with a `MetalGUI` so the geometry can be inspected interactively.

In [1]:
from collections import OrderedDict

from qiskit_metal import designs, Dict, MetalGUI
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.qubits.circle_transmon_squid import CircTransmonSQUID

from build_design import (
    L_TRANSMON_NH, L_PROBE_NH, L_BRIDGED_JJ_B_NH, CJ_BRIDGED_JJ_B_FF,
    CJ_TRANSMON_FF, CJ_PROBE_FF, QUBIT_COMPONENT_NAME,
    RES_LENGTH_MM, CHIP_SIZE_Z_UM, CHIP_MARGIN_FACTOR, _design_bounds_mm,
)

In [2]:
design = designs.DesignPlanar({}, overwrite_enabled=True)
gui = MetalGUI(design)

## Global CPW variables

The chip itself is sized *after* the geometry below is drawn, from the geometry's own bounding box (see the last cell) - so there's no fixed chip size to set here.

In [3]:
design.variables['cpw_width'] = '20um'
design.variables['cpw_gap'] = '12.25um'

## Qubit (circular transmon with SQUID)

SQUID branch `a` is the real junction gap (probe port, `L_PROBE_NH`); branch `b` is bridged with
continuous metal (`export_mask=True`) per `EPR_C_EXTRACTION.md` Sec 4.1 - it should render as a
single unbroken metal capsule, not two separate pieces.

In [4]:
qubit_options = Dict(
    pos_x='0um', pos_y='0um', orientation='0', chip='main',
    cpw_width='cpw_width', cpw_gap='cpw_gap',

    pad_radius='90um',
    pad_gap='10um',

    jj_options=Dict(
        jj_width='5um',
        jj_angle='0',
        L_j=f'{L_TRANSMON_NH}nH',
        C_j=f'{CJ_TRANSMON_FF}fF',
        export_mask=False,
        jj_sim_gap='3um',
    ),

    squid_options=Dict(
        theta_2='180',
        delta_angle='7.5',
        g1='4um',

        l1='600um',
        w1='6um',
        l2='600um',

        jj_a_options=Dict(
            jj_width='4um', L_j=f'{L_PROBE_NH}nH', C_j=f'{CJ_PROBE_FF}fF',
            export_mask=False, jj_sim_gap='3um',
        ),
        jj_b_options=Dict(
            jj_width='6um', L_j=f'{L_BRIDGED_JJ_B_NH}nH', C_j=f'{CJ_BRIDGED_JJ_B_FF}fF',
            export_mask=True, jj_sim_gap='3um',
        ),
    ),
)
qubit_1 = CircTransmonSQUID(design, QUBIT_COMPONENT_NAME, options=qubit_options)

gui.rebuild()
gui.autoscale()

## Readout resonator terminations

In [5]:
launch_point1 = OpenToGround(design, 'launch_point1', options=dict(
    pos_x='-275um', pos_y='-850um', orientation='0',
    termination_gap='cpw_gap', gap='cpw_gap', width='cpw_width'))

launch_point2 = ShortToGround(design, 'launch_point2', options=dict(
    pos_x='-150um', pos_y='50um', orientation='0',
    termination_gap='cpw_gap', gap='cpw_gap', width='cpw_width'))

gui.rebuild()
gui.autoscale()

## Meandered readout resonator

In [6]:
jogs = OrderedDict()
jogs[0] = ["L", "100um"]
jogs[1] = ["L", "700um"]

readout_1 = RouteMeander(design, 'readout_1', options=dict(
    pin_inputs=dict(
        start_pin=dict(component='launch_point1', pin='open'),
        end_pin=dict(component='launch_point2', pin='short'),
    ),
    fillet='40um',
    lead=dict(
        start_straight='10um',
        end_straight='1180um',
        end_jogged_extension=jogs,
    ),
    total_length=f'{RES_LENGTH_MM}mm',
))

gui.rebuild()
gui.autoscale()

## Size the chip from the drawn geometry's own bounding box

Everything above is drawn without regard to chip size (component coordinates don't depend on it). Now compute the bounding box of everything drawn and size the chip around it (`* CHIP_MARGIN_FACTOR`), so the chip footprint stays tight instead of being an arbitrary fixed guess.

In [7]:
minx, miny, maxx, maxy = _design_bounds_mm(design)
center_x, center_y = (minx + maxx) / 2, (miny + maxy) / 2
size_x, size_y = (maxx - minx) * CHIP_MARGIN_FACTOR, (maxy - miny) * CHIP_MARGIN_FACTOR

design.chips.main.size.size_x = f'{size_x}mm'
design.chips.main.size.size_y = f'{size_y}mm'
design.chips.main.size.size_z = f'{CHIP_SIZE_Z_UM}um'
design.chips.main.size.center_x = f'{center_x}mm'
design.chips.main.size.center_y = f'{center_y}mm'

print(f"Design bounds (mm): X=[{minx:.4f},{maxx:.4f}]  Y=[{miny:.4f},{maxy:.4f}]")
print(f"Chip size (mm): {size_x:.4f} x {size_y:.4f}, centered at ({center_x:.4f}, {center_y:.4f})")

gui.rebuild()
gui.autoscale()

Design bounds (mm): X=[-1.3300,0.1000]  Y=[-0.8722,0.1000]
Chip size (mm): 5.7200 x 3.8890, centered at (-0.6150, -0.3861)


In [8]:
print("Components:", list(design.components.keys()))

Components: ['Q1_SQUID', 'launch_point1', 'launch_point2', 'readout_1']
